In [2]:
# ============================================================
# IMPORTS
# ============================================================

import copy
import gc
import json
import math
import random
import warnings
from pathlib import Path
import joblib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    classification_report
)
from sklearn.preprocessing import StandardScaler, LabelEncoder

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

print("Imports OK")

Imports OK


In [3]:
# ============================================================
# REPRODUCIBILITY
# ============================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [4]:
# ============================================================
# PATHS AND GLOBAL CONFIG
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility"
)

DATA_DIR = PROJECT_DIR / "processed_data_final"

OUTPUT_DIR = DATA_DIR / "transformer_idm_feature_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# -------------------------
# REGIME THRESHOLDS
# -------------------------
BRAKE_THRESHOLD = -0.2
ACCEL_THRESHOLD = 0.2

THRESHOLD_TAG = f"brake{abs(BRAKE_THRESHOLD):.1f}_accel{ACCEL_THRESHOLD:.1f}".replace(".", "p")

print("Using regime definition:")
print(f"  braking      : a_true < {BRAKE_THRESHOLD}")
print(f"  accelerating : a_true > {ACCEL_THRESHOLD}")
print("  steady       : otherwise")

# -------------------------
# MODEL / TRAINING CONFIG
# -------------------------
HISTORY = 25
HORIZON = 1
STRIDE = 1

SEQUENCE_TAG = f"H{HISTORY}_Hz{HORIZON}_S{STRIDE}"

BATCH_SIZE = 128
EPOCHS = 10
LR = 1e-3

D_MODEL = 64
NHEAD = 4
NUM_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.1

# -------------------------
# EXPERIMENT CONFIG
# -------------------------
PRIOR_MODE = "xgb_best_idm_soft"
IDM_PARAM_TAG = "refinedIDM_v2"

EXPERIMENT_NAME = (
    f"TransformerIDMFeature_XGBSoftBestIDM_"
    f"{SEQUENCE_TAG}_{THRESHOLD_TAG}_{IDM_PARAM_TAG}"
)

RUN_DIR = OUTPUT_DIR / EXPERIMENT_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Experiment:", EXPERIMENT_NAME)
print("Prior mode:", PRIOR_MODE)
print("Sequence tag:", SEQUENCE_TAG)
print("Run dir:", RUN_DIR)
print("IDM parameter tag:", IDM_PARAM_TAG)

assert DATA_DIR.exists(), f"DATA_DIR does not exist: {DATA_DIR}"

DEVICE: cuda
Using regime definition:
  braking      : a_true < -0.2
  accelerating : a_true > 0.2
  steady       : otherwise
Data dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final
Output dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs
Experiment: TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2
Prior mode: xgb_best_idm_soft
Sequence tag: H25_Hz1_S1
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2
IDM parameter tag: refinedIDM_v2


In [5]:
# ============================================================
# LOAD PRE-SPLIT DATASETS
# Memory-safe: only load columns needed in this notebook
# ============================================================

required_files = [
    DATA_DIR / "episode_tracks_train.csv",
    DATA_DIR / "episode_tracks_val.csv",
    DATA_DIR / "episode_tracks_test.csv",
]

for p in required_files:
    assert p.exists(), f"Missing required file: {p}"


needed_cols = [
    "recording_id",
    "id",
    "unique_id",
    "episode_id",
    "frame",
    "xVelocity",
    "precedingId",
    "precedingXVelocity",
    "xAcceleration",
    "dhw",
    "split",
]


def read_episode_tracks_minimal(path):
    header = pd.read_csv(path, nrows=0).columns.tolist()
    usecols = [c for c in needed_cols if c in header]

    dtype_map = {
        "recording_id": "int32",
        "id": "int32",
        "frame": "int32",
        "precedingId": "float32",
        "xVelocity": "float32",
        "precedingXVelocity": "float32",
        "xAcceleration": "float32",
        "dhw": "float32",
    }

    dtype_map = {k: v for k, v in dtype_map.items() if k in usecols}

    df = pd.read_csv(
        path,
        usecols=usecols,
        dtype=dtype_map,
        low_memory=False,
    )

    if "unique_id" in df.columns:
        df["unique_id"] = df["unique_id"].astype("category")

    if "episode_id" in df.columns:
        df["episode_id"] = df["episode_id"].astype("category")

    if "split" in df.columns:
        df["split"] = df["split"].astype("category")

    return df


episode_tracks_train = read_episode_tracks_minimal(DATA_DIR / "episode_tracks_train.csv")
episode_tracks_val   = read_episode_tracks_minimal(DATA_DIR / "episode_tracks_val.csv")
episode_tracks_test  = read_episode_tracks_minimal(DATA_DIR / "episode_tracks_test.csv")

print("train:", episode_tracks_train.shape)
print("val  :", episode_tracks_val.shape)
print("test :", episode_tracks_test.shape)

# Split integrity check using only compact columns
all_loaded = pd.concat(
    [
        episode_tracks_train[["unique_id", "episode_id", "split"]],
        episode_tracks_val[["unique_id", "episode_id", "split"]],
        episode_tracks_test[["unique_id", "episode_id", "split"]],
    ],
    ignore_index=True
)

ego_split_check = all_loaded.groupby("unique_id")["split"].nunique()
episode_split_check = all_loaded.groupby("episode_id")["split"].nunique()

print("\nEgos in multiple splits:", (ego_split_check > 1).sum())
print("Episodes in multiple splits:", (episode_split_check > 1).sum())

assert (ego_split_check <= 1).all(), "Some egos appear in multiple splits."
assert (episode_split_check <= 1).all(), "Some episodes appear in multiple splits."

del all_loaded, ego_split_check, episode_split_check
gc.collect()

print("Split integrity checks passed.")

train: (11094352, 11)
val  : (2370107, 11)
test : (2368174, 11)

Egos in multiple splits: 0
Episodes in multiple splits: 0
Split integrity checks passed.


In [6]:
# ============================================================
# FEATURE PREPARATION
# Same base logic as IDM / Transformer notebooks
# Memory-safe: keeps only columns needed downstream
# ============================================================

def prepare_base_features(df):
    keep_cols = [
        "episode_id",
        "unique_id",
        "split",
        "frame",
        "xVelocity",
        "precedingId",
        "precedingXVelocity",
        "xAcceleration",
        "dhw",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    out = df[keep_cols].copy()

    # Direction-normalized speed features
    out["v_ego"] = out["xVelocity"].abs().astype(np.float32)
    out["v_leader"] = out["precedingXVelocity"].abs().astype(np.float32)

    # True signed acceleration target
    out["a_true"] = out["xAcceleration"].astype(np.float32)

    # Detect no-leader cases robustly
    no_leader = (
        out["precedingId"].isna()
        | (out["precedingId"] <= 0)
        | out["precedingXVelocity"].isna()
        | out["dhw"].isna()
    )

    # Relative speed and spacing
    out["delta_v"] = (out["v_ego"] - out["v_leader"]).astype(np.float32)
    out["gap"] = out["dhw"].clip(lower=0.1).astype(np.float32)

    # Free-road fallback for IDM-related calculations
    out.loc[no_leader, "gap"] = np.float32(1e6)
    out.loc[no_leader, "delta_v"] = np.float32(0.0)
    out.loc[no_leader, "v_leader"] = out.loc[no_leader, "v_ego"]

    out["no_leader"] = no_leader.astype("int8")

    # Keep only final columns needed from this point onward
    final_cols = [
        "episode_id",
        "unique_id",
        "split",
        "frame",
        "v_ego",
        "v_leader",
        "delta_v",
        "gap",
        "a_true",
        "no_leader",
    ]

    final_cols = [c for c in final_cols if c in out.columns]

    out = (
        out[final_cols]
        .sort_values(["episode_id", "frame"])
        .reset_index(drop=True)
    )

    return out


train_base = prepare_base_features(episode_tracks_train)
val_base   = prepare_base_features(episode_tracks_val)
test_base  = prepare_base_features(episode_tracks_test)

# Free raw loaded frames
del episode_tracks_train, episode_tracks_val, episode_tracks_test
gc.collect()

print(train_base[[
    "episode_id", "frame", "v_ego", "v_leader",
    "delta_v", "gap", "a_true", "no_leader"
]].head())

print("\nBase frame shapes:")
print("train_base:", train_base.shape)
print("val_base  :", val_base.shape)
print("test_base :", test_base.shape)

  episode_id  frame      v_ego   v_leader   delta_v        gap  a_true  \
0          1  21549  41.189999  43.099998 -1.910000  29.230000   -0.39   
1          1  21550  41.200001  43.130001 -1.930000  29.459999   -0.39   
2          1  21551  41.220001  43.150002 -1.930000  29.639999   -0.40   
3          1  21552  41.230000  43.180000 -1.950001  29.780001   -0.41   
4          1  21553  41.250000  43.200001 -1.950001  29.900000   -0.42   

   no_leader  
0          0  
1          0  
2          0  
3          0  
4          0  

Base frame shapes:
train_base: (11094352, 10)
val_base  : (2370107, 10)
test_base : (2368174, 10)


In [7]:
# ============================================================
# REGIME LABEL HELPER
# ============================================================
def add_regime_column(df, true_col="a_true"):
    out = df.copy()
    out["regime"] = np.where(
        out[true_col] < BRAKE_THRESHOLD, "braking",
        np.where(out[true_col] > ACCEL_THRESHOLD, "accelerating", "steady")
    )
    return out

In [8]:
# ============================================================
# IDM ACCELERATION
# ============================================================

def idm_acceleration(v, delta_v, s, params):
    v0 = max(float(params["v0"]), 1e-6)
    T = float(params["T"])
    s0 = float(params["s0"])
    a = max(float(params["a"]), 1e-6)
    b = max(float(params["b"]), 1e-6)
    delta = float(params["delta"])

    v = np.asarray(v, dtype=np.float32)
    delta_v = np.asarray(delta_v, dtype=np.float32)
    s = np.asarray(s, dtype=np.float32)

    v = np.maximum(v, 0.0)
    s = np.maximum(s, 0.1)

    sqrt_ab = np.sqrt(a * b)

    s_star = s0 + np.maximum(
        0.0,
        v * T + (v * delta_v) / (2.0 * sqrt_ab)
    )

    acc = a * (1.0 - (v / v0) ** delta - (s_star / s) ** 2)

    acc = np.clip(acc, -8.0, 4.0)

    return acc.astype(np.float32)

In [9]:
# ============================================================
# FINAL IDM PARAMETER SETS
# ============================================================

best_global_params = {"v0": 40.0, "T": 0.5,"s0": 0.5, "a": 0.4,"b": 5.0,"delta": 4.0}

best_params_braking = {'v0': 1.5, 'T': 0.3, 's0': 5.0, 'a': 0.1, 'b': 26.0, 'delta': 0.5}

best_params_accel = {'v0': 115.0, 'T': 0.1, 's0': 0.05, 'a': 0.4, 'b': 22.0, 'delta': 14.0}

best_params_steady = {'v0': 44.0, 'T': 0.3, 's0': 0.1, 'a': 0.1, 'b': 8.0, 'delta': 2.0}

In [10]:
# ============================================================
# BUILD XGB-SOFT BEST-IDM SELECTOR DATASET
# ============================================================

def add_selector_features(df):
    out = df.copy()

    out["time_headway"] = out["gap"] / np.maximum(out["v_ego"], 0.1)
    out["closing_speed"] = np.maximum(out["delta_v"], 0.0)
    out["opening_speed"] = np.maximum(-out["delta_v"], 0.0)
    out["abs_delta_v"] = np.abs(out["delta_v"])
    out["inv_ttc"] = out["closing_speed"] / np.maximum(out["gap"], 0.1)
    out["log_gap"] = np.log1p(np.minimum(out["gap"], 500.0))

    return out


def add_all_idm_candidates(df):
    out = df.copy()

    out["a_idm_global"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_global_params
    )

    out["a_idm_braking"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_params_braking
    )

    out["a_idm_accel"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_params_accel
    )

    out["a_idm_steady"] = idm_acceleration(
        out["v_ego"].values,
        out["delta_v"].values,
        out["gap"].values,
        best_params_steady
    )

    return out


def add_best_idm_label(df):
    out = df.copy()

    candidate_cols = [
        "a_idm_global",
        "a_idm_braking",
        "a_idm_accel",
        "a_idm_steady"
    ]

    errors = np.column_stack([
        np.abs(out["a_true"].values - out[col].values)
        for col in candidate_cols
    ])

    best_idx = np.argmin(errors, axis=1)

    label_names = np.array([
        "global",
        "braking",
        "accelerating",
        "steady"
    ])

    out["best_idm_label"] = label_names[best_idx]
    out["best_idm_error"] = errors[np.arange(len(out)), best_idx]

    return out


def build_xgb_selector_dataframe(df):
    out = df.copy()
    out = add_selector_features(out)
    out = add_all_idm_candidates(out)
    out = add_best_idm_label(out)

    return out


train_selector_df = build_xgb_selector_dataframe(train_base)
val_selector_df   = build_xgb_selector_dataframe(val_base)
test_selector_df  = build_xgb_selector_dataframe(test_base)

selector_cols = [
    "v_ego",
    "v_leader",
    "delta_v",
    "gap",
    "time_headway",
    "closing_speed",
    "opening_speed",
    "abs_delta_v",
    "inv_ttc",
    "log_gap"
]

print("Selector columns:")
print(selector_cols)

print("\nBest-IDM label counts on train:")
print(train_selector_df["best_idm_label"].value_counts())

print("\nBest-IDM label counts on val:")
print(val_selector_df["best_idm_label"].value_counts())

print("\nPreview:")
print(train_selector_df[
    selector_cols + [
        "a_true",
        "a_idm_global",
        "a_idm_braking",
        "a_idm_accel",
        "a_idm_steady",
        "best_idm_label",
        "best_idm_error"
    ]
].head())

Selector columns:
['v_ego', 'v_leader', 'delta_v', 'gap', 'time_headway', 'closing_speed', 'opening_speed', 'abs_delta_v', 'inv_ttc', 'log_gap']

Best-IDM label counts on train:
best_idm_label
steady          4934299
accelerating    2351734
global          2075564
braking         1732755
Name: count, dtype: int64

Best-IDM label counts on val:
best_idm_label
steady          1050192
accelerating     496232
global           454274
braking          369409
Name: count, dtype: int64

Preview:
       v_ego   v_leader   delta_v        gap  time_headway  closing_speed  \
0  41.189999  43.099998 -1.910000  29.230000      0.709638            0.0   
1  41.200001  43.130001 -1.930000  29.459999      0.715048            0.0   
2  41.220001  43.150002 -1.930000  29.639999      0.719068            0.0   
3  41.230000  43.180000 -1.950001  29.780001      0.722290            0.0   
4  41.250000  43.200001 -1.950001  29.900000      0.724848            0.0   

   opening_speed  abs_delta_v  inv_ttc   log

In [11]:
# ============================================================
# TRAIN XGBOOST BEST-IDM SELECTOR
# ============================================================

le_best = LabelEncoder()

y_train_xgb = le_best.fit_transform(train_selector_df["best_idm_label"])
y_val_xgb   = le_best.transform(val_selector_df["best_idm_label"])
y_test_xgb  = le_best.transform(test_selector_df["best_idm_label"])

xgb_best_selector = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist"
)

xgb_best_selector.fit(
    train_selector_df[selector_cols],
    y_train_xgb
)

val_pred_idx = xgb_best_selector.predict(val_selector_df[selector_cols])
val_pred_label = le_best.inverse_transform(val_pred_idx.astype(int))

print("=== XGB BEST-IDM SELECTOR VALIDATION REPORT ===")
print(classification_report(
    val_selector_df["best_idm_label"],
    val_pred_label,
    digits=4
))

print("XGB classes:")
print(le_best.classes_)

=== XGB BEST-IDM SELECTOR VALIDATION REPORT ===
              precision    recall  f1-score   support

accelerating     0.4808    0.5293    0.5039    496232
     braking     0.5604    0.3215    0.4086    369409
      global     0.3680    0.1261    0.1878    454274
      steady     0.5568    0.7721    0.6470   1050192

    accuracy                         0.5272   2370107
   macro avg     0.4915    0.4372    0.4368   2370107
weighted avg     0.5053    0.5272    0.4919   2370107

XGB classes:
['accelerating' 'braking' 'global' 'steady']


In [12]:
# ============================================================
# BUILD XGB-SOFT BEST-IDM PRIOR DATAFRAMES
# ============================================================

def add_xgb_soft_best_idm_prior(df, selector_model, label_encoder):
    out = df.copy()

    out = add_selector_features(out)
    out = add_all_idm_candidates(out)

    proba = selector_model.predict_proba(out[selector_cols])
    classes = list(label_encoder.classes_)

    class_to_col = {
        "global": "a_idm_global",
        "braking": "a_idm_braking",
        "accelerating": "a_idm_accel",
        "steady": "a_idm_steady"
    }

    out["a_idm_prior"] = 0.0

    for j, cls in enumerate(classes):
        out[f"p_{cls}"] = proba[:, j]
        out["a_idm_prior"] += proba[:, j] * out[class_to_col[cls]].values

    hard_idx = np.argmax(proba, axis=1)
    out["xgb_best_idm_hard_label"] = label_encoder.inverse_transform(hard_idx)

    return out


train_prior = add_xgb_soft_best_idm_prior(
    train_base,
    xgb_best_selector,
    le_best
)

val_prior = add_xgb_soft_best_idm_prior(
    val_base,
    xgb_best_selector,
    le_best
)

test_prior = add_xgb_soft_best_idm_prior(
    test_base,
    xgb_best_selector,
    le_best
)

print("Experiment:", EXPERIMENT_NAME)
print("Prior mode:", PRIOR_MODE)
print("Run dir:", RUN_DIR)

print("\nXGB-soft prior summary on test:")
print(test_prior["a_idm_prior"].describe())

print("\nHard label counts on test:")
print(test_prior["xgb_best_idm_hard_label"].value_counts())

prob_cols = [c for c in test_prior.columns if c.startswith("p_")]

print("\nMean XGB probabilities on test:")
print(test_prior[prob_cols].mean().sort_values(ascending=False))

print("\nPreview:")
print(test_prior[
    [
        "v_ego",
        "v_leader",
        "delta_v",
        "gap",
        "a_true",
        "a_idm_global",
        "a_idm_braking",
        "a_idm_accel",
        "a_idm_steady",
        "a_idm_prior",
        "xgb_best_idm_hard_label"
    ] + prob_cols
].head())

Experiment: TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2
Prior mode: xgb_best_idm_soft
Run dir: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2

XGB-soft prior summary on test:
count    2.368174e+06
mean     8.230635e-02
std      6.652159e-02
min     -4.226514e+00
25%      6.517071e-02
50%      8.098131e-02
75%      1.195554e-01
max      3.792363e-01
Name: a_idm_prior, dtype: float64

Hard label counts on test:
xgb_best_idm_hard_label
steady          1468305
accelerating     542283
braking          203909
global           153677
Name: count, dtype: int64

Mean XGB probabilities on test:
p_steady          0.448168
p_accelerating    0.209425
p_global          0.187750
p_braking         0.154656
dtype: float32

Preview:
       v_ego  v_leader   delta_v        gap  a_true  a_idm_global  \
0  23.209999     23.27 -0.0600

In [13]:
# ============================================================
# BUILD SEQUENCES WITH IDM PRIOR AS AN INPUT FEATURE
# ============================================================

FEATURE_COLS = ["v_ego", "v_leader", "delta_v", "gap", "a_idm_prior"]

def make_sequences_with_idm_feature(df, history=HISTORY, horizon=HORIZON, stride=STRIDE):
    X_seq = []
    y_target = []
    meta_rows = []

    for ep_id, g in df.groupby("episode_id"):
        g = g.sort_values("frame").reset_index(drop=True)

        values = g[FEATURE_COLS].values.astype(np.float32)
        a_true = g["a_true"].values.astype(np.float32)

        for t in range(history - 1, len(g) - horizon, stride):
            hist_slice = slice(t - history + 1, t + 1)
            target_idx = t + horizon

            X_seq.append(values[hist_slice])
            y_target.append(a_true[target_idx])

            row = {
                "episode_id": ep_id,
                "unique_id": g.loc[target_idx, "unique_id"] if "unique_id" in g.columns else np.nan,
                "split": g.loc[target_idx, "split"] if "split" in g.columns else np.nan,
                "frame": g.loc[target_idx, "frame"],
                "a_true": a_true[target_idx],
                "a_idm_prior": g.loc[target_idx, "a_idm_prior"],
                "v_ego": g.loc[target_idx, "v_ego"],
                "v_leader": g.loc[target_idx, "v_leader"],
                "delta_v": g.loc[target_idx, "delta_v"],
                "gap": g.loc[target_idx, "gap"],
            }

            # keep candidate IDM values and XGB-soft information for later analysis
            for col in [
                "a_idm_global",
                "a_idm_braking",
                "a_idm_accel",
                "a_idm_steady",
                "xgb_best_idm_hard_label",
                "p_global",
                "p_braking",
                "p_accelerating",
                "p_steady"
            ]:
                if col in g.columns:
                    row[col] = g.loc[target_idx, col]

            meta_rows.append(row)

    X_seq = np.array(X_seq, dtype=np.float32)
    y_target = np.array(y_target, dtype=np.float32).reshape(-1, 1)
    meta_df = pd.DataFrame(meta_rows)

    assert len(X_seq) == len(y_target) == len(meta_df), (
        f"Length mismatch: X={len(X_seq)}, y={len(y_target)}, meta={len(meta_df)}"
    )

    return X_seq, y_target, meta_df


X_train, y_train, meta_train = make_sequences_with_idm_feature(
    train_prior,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE
)

X_val, y_val, meta_val = make_sequences_with_idm_feature(
    val_prior,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE
)

X_test, y_test, meta_test = make_sequences_with_idm_feature(
    test_prior,
    history=HISTORY,
    horizon=HORIZON,
    stride=STRIDE
)

print("Feature columns:", FEATURE_COLS)
print("X_train:", X_train.shape, "y_train:", y_train.shape, "meta_train:", meta_train.shape)
print("X_val  :", X_val.shape, "y_val:", y_val.shape, "meta_val:", meta_val.shape)
print("X_test :", X_test.shape, "y_test:", y_test.shape, "meta_test:", meta_test.shape)

assert len(meta_test) == len(y_test), "meta_test must be sequence-level, not frame-level"

Feature columns: ['v_ego', 'v_leader', 'delta_v', 'gap', 'a_idm_prior']
X_train: (10222602, 25, 5) y_train: (10222602, 1) meta_train: (10222602, 19)
X_val  : (2183357, 25, 5) y_val: (2183357, 1) meta_val: (2183357, 19)
X_test : (2181399, 25, 5) y_test: (2181399, 1) meta_test: (2181399, 19)


In [14]:
# ============================================================
# FREE FRAME-LEVEL PRIOR / SELECTOR DATAFRAMES AFTER SEQUENCES
# ============================================================

# From this point onward, the model uses:
# - X_train, X_val, X_test
# - y_train, y_val, y_test
# - meta_train, meta_val, meta_test
#
# The frame-level prior and selector dataframes are no longer needed.

for name in [
    "train_selector_df", "val_selector_df", "test_selector_df",
    "train_prior", "val_prior", "test_prior",
    "train_base", "val_base", "test_base",
]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Freed frame-level prior and selector dataframes.")

Freed frame-level prior and selector dataframes.


In [15]:
# ============================================================
# SANITY CHECK: IDM PRIOR IS INCLUDED AS A FEATURE
# ============================================================

print("Feature columns used by the Transformer:")
print(FEATURE_COLS)

assert "a_idm_prior" in FEATURE_COLS, "ERROR: a_idm_prior is not included in the feature columns."

idm_feature_index = FEATURE_COLS.index("a_idm_prior")

print("Index of a_idm_prior in input feature vector:", idm_feature_index)
print("Example raw IDM prior values from first training sequence:")
print(X_train[0, :, idm_feature_index][:5])

Feature columns used by the Transformer:
['v_ego', 'v_leader', 'delta_v', 'gap', 'a_idm_prior']
Index of a_idm_prior in input feature vector: 4
Example raw IDM prior values from first training sequence:
[0.13676624 0.14704189 0.145861   0.14586867 0.14681557]


In [16]:
# ============================================================
# SCALE INPUTS AND TARGETS
# ============================================================

X_scaler = StandardScaler()
y_scaler = StandardScaler()

# Fit X scaler on flattened train sequences
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_scaler.fit(X_train_2d)

def scale_X(X, scaler):
    n, h, d = X.shape
    X_2d = X.reshape(-1, d)
    X_2d_scaled = scaler.transform(X_2d)
    return X_2d_scaled.reshape(n, h, d).astype(np.float32)

X_train_s = scale_X(X_train, X_scaler)
X_val_s   = scale_X(X_val, X_scaler)
X_test_s  = scale_X(X_test, X_scaler)

y_scaler.fit(y_train)

y_train_s = y_scaler.transform(y_train).astype(np.float32)
y_val_s   = y_scaler.transform(y_val).astype(np.float32)
y_test_s  = y_scaler.transform(y_test).astype(np.float32)

# Save input dimension before freeing anything
INPUT_DIM = X_train_s.shape[-1]

print("Scaled inputs and targets.")
print("Input dimension:", INPUT_DIM)

# Free unscaled sequence arrays after scaling
del X_train, X_val, X_test
del X_train_2d
gc.collect()

print("Freed unscaled sequence arrays.")

Scaled inputs and targets.
Input dimension: 5
Freed unscaled sequence arrays.


In [17]:
# ============================================================
# DATASET
# ============================================================

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = SequenceDataset(X_train_s, y_train_s)
val_ds   = SequenceDataset(X_val_s, y_val_s)
test_ds  = SequenceDataset(X_test_s, y_test_s)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

# Free scaled NumPy arrays after tensors are created
del X_train_s, X_val_s, X_test_s
del y_train_s, y_val_s, y_test_s
gc.collect()

print("Datasets and loaders ready.")
print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))
print("Test samples :", len(test_ds))

Datasets and loaders ready.
Train samples: 10222602
Val samples  : 2183357
Test samples : 2181399


In [18]:
# ============================================================
# TRANSFORMER MODEL WITH IDM FEATURE INPUT
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerIDMFeatureRegressor(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 1)
        )

    def forward(self, x):
        h = self.input_proj(x)
        h = self.pos_encoder(h)
        h = self.encoder(h)

        h_last = h[:, -1, :]
        y = self.head(h_last)

        return y

In [19]:
# ============================================================
# TRAIN / EVALUATE UTILITIES
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()

    losses = []
    preds = []
    targets = []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)

        pred = model(Xb)
        loss = criterion(pred, yb)

        losses.append(loss.item())
        preds.append(pred.cpu().numpy())
        targets.append(yb.cpu().numpy())

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    return float(np.mean(losses)), preds, targets

In [20]:
# ============================================================
# TRAIN TRANSFORMER WITH IDM FEATURE
# Saves best checkpoint immediately and last checkpoint every epoch.
# ============================================================

ckpt_path = RUN_DIR / f"transformer_xgb_soft_best_idm_feature_best_{SEQUENCE_TAG}.pt"
last_ckpt_path = RUN_DIR / f"transformer_xgb_soft_best_idm_feature_last_{SEQUENCE_TAG}.pt"
log_path = RUN_DIR / f"transformer_xgb_soft_best_idm_feature_training_log_{SEQUENCE_TAG}.csv"

FORCE_RETRAIN = True

print("Experiment:", EXPERIMENT_NAME)
print("Prior mode:", PRIOR_MODE)
print("Brake threshold:", BRAKE_THRESHOLD)
print("Accel threshold:", ACCEL_THRESHOLD)
print("Sequence tag:", SEQUENCE_TAG)
print("Best checkpoint path:", ckpt_path)
print("Last checkpoint path:", last_ckpt_path)
print("Log path:", log_path)

if FORCE_RETRAIN:
    for p in [ckpt_path, last_ckpt_path, log_path]:
        if p.exists():
            p.unlink()
            print("Deleted old file:", p)

model = TransformerIDMFeatureRegressor(
    input_dim=INPUT_DIM,
    d_model=D_MODEL,
    nhead=NHEAD,
    num_layers=NUM_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_loss = float("inf")
best_epoch = -1
best_state = None

train_losses = []
val_losses = []
log_rows = []

if ckpt_path.exists() and not FORCE_RETRAIN:
    print("Loading existing checkpoint...")
    checkpoint = torch.load(ckpt_path, map_location=DEVICE)

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        best_val_loss = checkpoint.get("best_val_loss", np.nan)
        best_epoch = checkpoint.get("best_epoch", -1)
        train_losses = checkpoint.get("train_losses", [])
        val_losses = checkpoint.get("val_losses", [])
    else:
        model.load_state_dict(checkpoint)

else:
    print("Training Transformer with IDM feature...")

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            DEVICE
        )

        val_loss, _, _ = evaluate_model(
            model,
            val_loader,
            criterion,
            DEVICE
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        improved = val_loss < best_val_loss

        if improved:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

            torch.save(
                {
                    "model_state_dict": best_state,
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_loss": best_val_loss,
                    "best_epoch": best_epoch,
                    "train_losses": train_losses,
                    "val_losses": val_losses,
                    "experiment_name": EXPERIMENT_NAME,
                    "prior_mode": PRIOR_MODE,
                    "sequence_tag": SEQUENCE_TAG,
                    "feature_cols": FEATURE_COLS,
                    "HISTORY": HISTORY,
                    "HORIZON": HORIZON,
                    "STRIDE": STRIDE,
                    "model_config": {
                        "input_dim": INPUT_DIM,
                        "d_model": D_MODEL,
                        "nhead": NHEAD,
                        "num_layers": NUM_LAYERS,
                        "dim_feedforward": DIM_FEEDFORWARD,
                        "dropout": DROPOUT,
                    }
                },
                ckpt_path
            )

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
                "completed_epoch": epoch,
                "train_losses": train_losses,
                "val_losses": val_losses,
                "experiment_name": EXPERIMENT_NAME,
                "prior_mode": PRIOR_MODE,
                "idm_param_tag": IDM_PARAM_TAG,
                "sequence_tag": SEQUENCE_TAG,
                "feature_cols": FEATURE_COLS,
                "HISTORY": HISTORY,
                "HORIZON": HORIZON,
                "STRIDE": STRIDE,
                "model_config": {
                    "input_dim": INPUT_DIM,
                    "d_model": D_MODEL,
                    "nhead": NHEAD,
                    "num_layers": NUM_LAYERS,
                    "dim_feedforward": DIM_FEEDFORWARD,
                    "dropout": DROPOUT,
                },
            },
            last_ckpt_path
        )

        log_rows.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "improved": improved,
        })

        pd.DataFrame(log_rows).to_csv(log_path, index=False)

        status = " | improved + saved best" if improved else ""

        print(
            f"[Epoch {epoch:02d}] "
            f"train_loss={train_loss:.6f} "
            f"val_loss={val_loss:.6f}{status}"
        )

    # Load best checkpoint after training
    best_checkpoint = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(best_checkpoint["model_state_dict"])

print("Transformer with IDM feature ready.")
print("Best checkpoint:", ckpt_path)
print("Last checkpoint:", last_ckpt_path)
print("Training log:", log_path)
print("Best epoch:", best_epoch)
print("Best val loss:", best_val_loss)

Experiment: TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2
Prior mode: xgb_best_idm_soft
Brake threshold: -0.2
Accel threshold: 0.2
Sequence tag: H25_Hz1_S1
Best checkpoint path: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2\transformer_xgb_soft_best_idm_feature_best_H25_Hz1_S1.pt
Last checkpoint path: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2\transformer_xgb_soft_best_idm_feature_last_H25_Hz1_S1.pt
Log path: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2\transformer_xgb_soft_best_idm_feature_training_log_H25_Hz1_S1.csv
Training Transf

In [21]:
# ============================================================
# FINAL PREDICTION
# ============================================================

_, train_pred_s, train_true_s = evaluate_model(model, train_loader, criterion, DEVICE)
_, val_pred_s, val_true_s = evaluate_model(model, val_loader, criterion, DEVICE)
_, test_pred_s, test_true_s = evaluate_model(model, test_loader, criterion, DEVICE)

train_pred = y_scaler.inverse_transform(train_pred_s)
val_pred   = y_scaler.inverse_transform(val_pred_s)
test_pred  = y_scaler.inverse_transform(test_pred_s)

train_true = y_scaler.inverse_transform(train_true_s)
val_true   = y_scaler.inverse_transform(val_true_s)
test_true  = y_scaler.inverse_transform(test_true_s)

In [22]:
# ============================================================
# METRIC HELPERS
# ============================================================

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    medae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    bias = float(np.mean(y_pred - y_true))
    corr = float(np.corrcoef(y_true, y_pred)[0, 1]) if len(y_true) > 1 else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "MedAE": medae,
        "R2": r2,
        "Bias": bias,
        "Corr": corr
    }


metrics_train = compute_metrics(train_true, train_pred)
metrics_val   = compute_metrics(val_true, val_pred)
metrics_test  = compute_metrics(test_true, test_pred)

split_metrics_df = pd.DataFrame([
    {"split": "train", **metrics_train},
    {"split": "val", **metrics_val},
    {"split": "test", **metrics_test},
])

print("\n=== TRANSFORMER + IDM FEATURE METRICS BY SPLIT ===")
print(split_metrics_df.to_string(index=False))


=== TRANSFORMER + IDM FEATURE METRICS BY SPLIT ===
split      MAE     RMSE    MedAE       R2      Bias     Corr
train 0.147629 0.266055 0.078565 0.361531 -0.010301 0.603041
  val 0.149451 0.268511 0.080432 0.332461 -0.010830 0.579847
 test 0.148338 0.268491 0.077588 0.332320 -0.012505 0.579028


In [23]:
# ============================================================
# FINAL LENGTH CHECK BEFORE EVALUATION
# ============================================================

print("len(meta_test):", len(meta_test))
print("len(test_true):", len(test_true))
print("len(test_pred):", len(test_pred))

assert len(meta_test) == len(test_true) == len(test_pred), (
    f"Length mismatch: meta_test={len(meta_test)}, "
    f"test_true={len(test_true)}, test_pred={len(test_pred)}"
)

len(meta_test): 2181399
len(test_true): 2181399
len(test_pred): 2181399


In [24]:
# ============================================================
# TEST RESULTS DATAFRAME AND REGIME EVALUATION
# ============================================================

test_results = meta_test.copy()

test_results["y_true"] = test_true.reshape(-1)
test_results["y_pred"] = test_pred.reshape(-1)

test_results["error"] = test_results["y_pred"] - test_results["y_true"]

test_results["abs_error"] = test_results["error"].abs()
test_results["Model"] = "Transformer + XGB-soft best-IDM feature"
test_results["PriorMode"] = PRIOR_MODE

test_results["regime"] = np.where(
    test_results["y_true"] < BRAKE_THRESHOLD, "braking",
    np.where(
        test_results["y_true"] > ACCEL_THRESHOLD,
        "accelerating",
        "steady"
    )
)

regime_metrics_df = (
    test_results.groupby("regime")
    .apply(lambda g: pd.Series({
        "MAE": np.mean(np.abs(g["y_true"] - g["y_pred"])),
        "RMSE": np.sqrt(np.mean((g["y_true"] - g["y_pred"]) ** 2)),
        "Bias": np.mean(g["y_pred"] - g["y_true"]),
        "Count": len(g)
    }))
    .reset_index()
)

regime_order = ["accelerating", "braking", "steady"]
regime_metrics_df["regime"] = pd.Categorical(
    regime_metrics_df["regime"],
    categories=regime_order,
    ordered=True
)

regime_metrics_df = regime_metrics_df.sort_values("regime").reset_index(drop=True)

print("\n=== TRANSFORMER + XGB-SOFT BEST-IDM FEATURE TEST METRICS BY REGIME ===")
print(regime_metrics_df.to_string(index=False))

if "xgb_best_idm_hard_label" in test_results.columns:
    print("\n=== XGB HARD LABEL COUNTS ON TEST SEQUENCES ===")
    print(test_results["xgb_best_idm_hard_label"].value_counts())

if "xgb_best_idm_hard_label" in test_results.columns:
    print("\n=== XGB HARD LABEL COUNTS BY TRUE REGIME ===")
    print(pd.crosstab(
        test_results["regime"],
        test_results["xgb_best_idm_hard_label"],
        normalize="index"
    ))

prob_cols = [c for c in test_results.columns if c.startswith("p_")]

if len(prob_cols) > 0:
    print("\n=== MEAN XGB SOFT PROBABILITIES ON TEST SEQUENCES ===")
    print(test_results[prob_cols].mean().sort_values(ascending=False))


=== TRANSFORMER + XGB-SOFT BEST-IDM FEATURE TEST METRICS BY REGIME ===
      regime      MAE     RMSE      Bias     Count
accelerating 0.208295 0.308666 -0.205011  623803.0
     braking 0.425611 0.577357  0.392185  264219.0
      steady 0.062778 0.086795 -0.002331 1293377.0

=== XGB HARD LABEL COUNTS ON TEST SEQUENCES ===
xgb_best_idm_hard_label
steady          1351584
accelerating     496535
braking          197478
global           135802
Name: count, dtype: int64

=== XGB HARD LABEL COUNTS BY TRUE REGIME ===
xgb_best_idm_hard_label  accelerating   braking    global    steady
regime                                                             
accelerating                 0.355046  0.061992  0.082290  0.500671
braking                      0.281872  0.282523  0.033098  0.402507
steady                       0.155082  0.065069  0.058548  0.721301

=== MEAN XGB SOFT PROBABILITIES ON TEST SEQUENCES ===
p_steady          0.447546
p_accelerating    0.208363
p_global          0.186428
p_braki

In [25]:
# ============================================================
# FINAL SUMMARY TABLE
# ============================================================

model_label = "Transformer + XGB-soft best-IDM feature"

summary_df = pd.DataFrame([
    {
        "Model": model_label,
        "PriorMode": PRIOR_MODE,
        "BrakeThreshold": BRAKE_THRESHOLD,
        "AccelThreshold": ACCEL_THRESHOLD,
        "Features": ", ".join(FEATURE_COLS),
        **metrics_test
    }
])

print("\n=== FINAL TRANSFORMER + XGB-SOFT BEST-IDM FEATURE SUMMARY ===")
print(summary_df.to_string(index=False))


=== FINAL TRANSFORMER + XGB-SOFT BEST-IDM FEATURE SUMMARY ===
                                  Model         PriorMode  BrakeThreshold  AccelThreshold                                   Features      MAE     RMSE    MedAE      R2      Bias     Corr
Transformer + XGB-soft best-IDM feature xgb_best_idm_soft            -0.2             0.2 v_ego, v_leader, delta_v, gap, a_idm_prior 0.148338 0.268491 0.077588 0.33232 -0.012505 0.579028


In [26]:
# ============================================================
# QUICK COMPARISON WITH PLAIN TRANSFORMER
# ============================================================

comparison_df = pd.DataFrame([
    {
        "Model": "Plain Transformer",
        "MAE": 0.143432,
        "RMSE": 0.265859,
        "MedAE": 0.071500,
        "R2": 0.345347,
        "Bias": -0.002777,
        "Corr": 0.589217
    },
    {
        "Model": model_label,
        "MAE": metrics_test["MAE"],
        "RMSE": metrics_test["RMSE"],
        "MedAE": metrics_test["MedAE"],
        "R2": metrics_test["R2"],
        "Bias": metrics_test["Bias"],
        "Corr": metrics_test["Corr"]
    }
])

print("\n=== PLAIN TRANSFORMER VS TRANSFORMER + XGB-SOFT BEST-IDM FEATURE ===")
print(comparison_df.to_string(index=False))


=== PLAIN TRANSFORMER VS TRANSFORMER + XGB-SOFT BEST-IDM FEATURE ===
                                  Model      MAE     RMSE    MedAE       R2      Bias     Corr
                      Plain Transformer 0.143432 0.265859 0.071500 0.345347 -0.002777 0.589217
Transformer + XGB-soft best-IDM feature 0.148338 0.268491 0.077588 0.332320 -0.012505 0.579028


In [27]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

summary_df.to_csv(
    RUN_DIR / "transformer_xgb_soft_best_idm_feature_summary.csv",
    index=False
)

summary_df.to_csv(
    RUN_DIR / f"transformer_xgb_soft_best_idm_feature_summary_{SEQUENCE_TAG}.csv",
    index=False
)

split_metrics_df.to_csv(
    RUN_DIR / "transformer_xgb_soft_best_idm_feature_split_metrics.csv",
    index=False
)

split_metrics_df.to_csv(
    RUN_DIR / f"transformer_xgb_soft_best_idm_feature_split_metrics_{SEQUENCE_TAG}.csv",
    index=False
)

regime_metrics_df.to_csv(
    RUN_DIR / "transformer_xgb_soft_best_idm_feature_regime_metrics.csv",
    index=False
)

regime_metrics_df.to_csv(
    RUN_DIR / f"transformer_xgb_soft_best_idm_feature_regime_metrics_{SEQUENCE_TAG}.csv",
    index=False
)

# Standard filename searched by secondary evaluation
test_results.to_csv(
    RUN_DIR / "transformer_xgb_soft_best_idm_feature_test_predictions.csv",
    index=False
)

# Extra sequence-specific filename for clarity
test_results.to_csv(
    RUN_DIR / f"transformer_xgb_soft_best_idm_feature_test_predictions_{SEQUENCE_TAG}.csv",
    index=False
)

# Optional extra alias, in case your secondary discovery list uses this shorter name
test_results.to_csv(
    RUN_DIR / "transformer_idm_feature_test_predictions.csv",
    index=False
)

joblib.dump(
    {
        "model": xgb_best_selector,
        "label_encoder": le_best,
        "selector_cols": selector_cols,
        "classes": list(le_best.classes_),
        "idm_param_tag": IDM_PARAM_TAG,
        "threshold_tag": THRESHOLD_TAG,
        "sequence_tag": SEQUENCE_TAG,
    },
    RUN_DIR / "xgb_soft_best_idm_selector_bundle.joblib"
)

print("Saved XGB-soft selector bundle:")
print(RUN_DIR / "xgb_soft_best_idm_selector_bundle.joblib")

with open(RUN_DIR / "config.json", "w") as f:
    json.dump({
        "EXPERIMENT_NAME": EXPERIMENT_NAME,
        "PRIOR_MODE": PRIOR_MODE,
        "IDM_PARAM_TAG": IDM_PARAM_TAG,
        "SEQUENCE_TAG": SEQUENCE_TAG,
        "BRAKE_THRESHOLD": BRAKE_THRESHOLD,
        "ACCEL_THRESHOLD": ACCEL_THRESHOLD,
        "HISTORY": HISTORY,
        "HORIZON": HORIZON,
        "STRIDE": STRIDE,
        "BATCH_SIZE": BATCH_SIZE,
        "EPOCHS": EPOCHS,
        "LR": LR,
        "D_MODEL": D_MODEL,
        "NHEAD": NHEAD,
        "NUM_LAYERS": NUM_LAYERS,
        "DIM_FEEDFORWARD": DIM_FEEDFORWARD,
        "DROPOUT": DROPOUT,
        "FEATURE_COLS": FEATURE_COLS,
        "selector_cols": selector_cols,
        "xgb_classes": list(le_best.classes_),
        "DATA_DIR": str(DATA_DIR),
        "OUTPUT_DIR": str(OUTPUT_DIR),
        "RUN_DIR": str(RUN_DIR),
        "best_global_params": best_global_params,
        "best_params_braking": best_params_braking,
        "best_params_accel": best_params_accel,
        "best_params_steady": best_params_steady,
    }, f, indent=2)

print("Saved outputs to:")
print(RUN_DIR)

print("\nPrediction files:")
print(RUN_DIR / "transformer_xgb_soft_best_idm_feature_test_predictions.csv")
print(RUN_DIR / f"transformer_xgb_soft_best_idm_feature_test_predictions_{SEQUENCE_TAG}.csv")
print(RUN_DIR / "transformer_idm_feature_test_predictions.csv")

Saved XGB-soft selector bundle:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2\xgb_soft_best_idm_selector_bundle.joblib
Saved outputs to:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2

Prediction files:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2\transformer_xgb_soft_best_idm_feature_test_predictions.csv
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\transformer_idm_feature_runs\TransformerIDMFeature_XGBSoftBestIDM_H25_Hz1_S1_brake0p2_accel0p2_refinedIDM_v2\transformer_xgb_soft_best_idm_feature_test_predictions_H25_Hz1_S1.csv
C:\Users\ANAR